# ARC Prize 2026 - ARC-AGI-2

Offline, model-proposed program synthesis with exact symbolic verification and leave-one-example-out calibration of direct grid predictions. Attach the [Qwen3-30B-A3B-Instruct-2507 Kaggle Model](https://www.kaggle.com/models/qwen-lm/qwen-3/Transformers/30b-a3b-instruct-2507) to the notebook before submission. The source package is embedded by `kaggle/build_submission_notebook.py`.


In [ ]:
from pathlib import Path
import sys

PACKAGE_DIR = Path('/kaggle/working/arc_solver')
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
SOURCES = {'__init__.py': '"""ARC-AGI-2 task-level transformation inference."""\n\nfrom .inference import SolveResult, solve_task\n\n__all__ = ["SolveResult", "solve_task"]\n', 'grids.py': '"""Grid validation and small immutable-grid utilities."""\n\nfrom __future__ import annotations\n\nfrom collections import Counter\nfrom typing import TypeAlias\n\nGrid: TypeAlias = tuple[tuple[int, ...], ...]\nRawGrid: TypeAlias = list[list[int]]\n\n\ndef as_grid(value: object) -> Grid:\n    """Validate a JSON-compatible ARC grid and return an immutable copy."""\n    if not isinstance(value, (list, tuple)) or not value:\n        raise ValueError("A grid must be a non-empty list of rows.")\n    rows: list[tuple[int, ...]] = []\n    width: int | None = None\n    for row in value:\n        if not isinstance(row, (list, tuple)) or not row:\n            raise ValueError("Grid rows must be non-empty lists.")\n        converted: list[int] = []\n        for cell in row:\n            if isinstance(cell, bool) or not isinstance(cell, int) or not 0 <= cell <= 9:\n                raise ValueError(f"Invalid ARC cell value: {cell!r}")\n            converted.append(cell)\n        if width is None:\n            width = len(converted)\n        elif len(converted) != width:\n            raise ValueError("Grid rows must all have the same width.")\n        rows.append(tuple(converted))\n    if len(rows) > 30 or width is None or width > 30:\n        raise ValueError("ARC grids must be no larger than 30 by 30.")\n    return tuple(rows)\n\n\ndef to_json_grid(grid: Grid) -> RawGrid:\n    return [list(row) for row in grid]\n\n\ndef shape(grid: Grid) -> tuple[int, int]:\n    return len(grid), len(grid[0])\n\n\ndef most_common_color(grid: Grid) -> int:\n    counts = Counter(cell for row in grid for cell in row)\n    return min(counts, key=lambda color: (-counts[color], color))\n\n\ndef crop(grid: Grid, top: int, left: int, bottom: int, right: int) -> Grid:\n    return tuple(row[left:right] for row in grid[top:bottom])\n\n\ndef bounding_box(grid: Grid, background: int) -> tuple[int, int, int, int] | None:\n    cells = [\n        (r, c)\n        for r, row in enumerate(grid)\n        for c, color in enumerate(row)\n        if color != background\n    ]\n    if not cells:\n        return None\n    rows, cols = zip(*cells)\n    return min(rows), min(cols), max(rows) + 1, max(cols) + 1\n', 'operations.py': '"""Small, data-independent transformations used by the synthesis baseline."""\n\nfrom __future__ import annotations\n\nfrom collections import Counter, deque\nfrom dataclasses import dataclass\nfrom functools import lru_cache\nfrom typing import Callable\n\nfrom .grids import Grid, bounding_box, crop, most_common_color, shape\n\nGridTransform = Callable[[Grid], Grid | None]\n\n\n@dataclass(frozen=True)\nclass Operation:\n    name: str\n    apply: GridTransform\n    cost: int = 1\n\n\ndef _rotate(grid: Grid, turns: int) -> Grid:\n    result = grid\n    for _ in range(turns):\n        result = tuple(tuple(row) for row in zip(*result[::-1]))\n    return result\n\n\ndef _flip_horizontal(grid: Grid) -> Grid:\n    return tuple(tuple(reversed(row)) for row in grid)\n\n\ndef _flip_vertical(grid: Grid) -> Grid:\n    return tuple(reversed(grid))\n\n\ndef _translate(grid: Grid, dx: int, dy: int) -> Grid:\n    height, width = shape(grid)\n    background = most_common_color(grid)\n    rows = [[background] * width for _ in range(height)]\n    for r in range(height):\n        for c in range(width):\n            nr, nc = r + dy, c + dx\n            if 0 <= nr < height and 0 <= nc < width:\n                rows[nr][nc] = grid[r][c]\n    return tuple(tuple(row) for row in rows)\n\n\ndef _infer_translations(examples: list[tuple[Grid, Grid]]) -> list[tuple[int, int]]:\n    source, target = examples[0]\n    if shape(source) != shape(target):\n        return []\n    background = most_common_color(source)\n    anchors = [\n        (r, c, source[r][c])\n        for r, row in enumerate(source)\n        for c in range(len(row))\n        if source[r][c] != background\n    ]\n    if not anchors:\n        return []\n    color_counts = Counter(color for _, _, color in anchors)\n    anchor_color = min(color_counts, key=lambda color: (color_counts[color], color))\n    anchor_row, anchor_col, _ = next(item for item in anchors if item[2] == anchor_color)\n    target_positions = [\n        (r, c)\n        for r, row in enumerate(target)\n        for c, color in enumerate(row)\n        if color == anchor_color\n    ]\n    return sorted(\n        (col - anchor_col, row - anchor_row)\n        for row, col in target_positions\n        if (row, col) != (anchor_row, anchor_col)\n        and all(\n            _translate(example, col - anchor_col, row - anchor_row) == expected\n            for example, expected in examples\n        )\n    )\n\n\ndef _background_candidates(examples: list[tuple[Grid, Grid]]) -> list[int]:\n    candidates = {0}\n    for grid, _ in examples:\n        candidates.add(most_common_color(grid))\n        border = (\n            list(grid[0])\n            + list(grid[-1])\n            + [row[0] for row in grid[1:-1]]\n            + [row[-1] for row in grid[1:-1]]\n        )\n        if border:\n            counts = Counter(border)\n            candidates.add(min(counts, key=lambda color: (-counts[color], color)))\n    return sorted(candidates)\n\n\n@lru_cache(maxsize=256)\ndef _isolate_color(grid: Grid, color: int) -> Grid:\n    background = most_common_color(grid)\n    return tuple(\n        tuple(cell if cell == color else background for cell in row)\n        for row in grid\n    )\n\n\ndef _remove_color(grid: Grid, color: int) -> Grid:\n    background = most_common_color(grid)\n    return tuple(\n        tuple(background if cell == color else cell for cell in row)\n        for row in grid\n    )\n\n\n@lru_cache(maxsize=256)\ndef _components(\n    grid: Grid,\n    background: int,\n) -> tuple[tuple[int, int, int, int, frozenset[tuple[int, int]]], ...]:\n    height, width = shape(grid)\n    seen: set[tuple[int, int]] = set()\n    components: list[tuple[int, int, int, int, frozenset[tuple[int, int]]]] = []\n    for r in range(height):\n        for c in range(width):\n            color = grid[r][c]\n            if color == background or (r, c) in seen:\n                continue\n            queue = deque([(r, c)])\n            seen.add((r, c))\n            cells: set[tuple[int, int]] = set()\n            while queue:\n                cr, cc = queue.popleft()\n                cells.add((cr, cc))\n                for nr, nc in ((cr - 1, cc), (cr + 1, cc), (cr, cc - 1), (cr, cc + 1)):\n                    if (\n                        0 <= nr < height\n                        and 0 <= nc < width\n                        and grid[nr][nc] == color\n                        and (nr, nc) not in seen\n                    ):\n                        seen.add((nr, nc))\n                        queue.append((nr, nc))\n            rows, cols = zip(*cells)\n            components.append(\n                (color, len(cells), min(rows), min(cols), frozenset(cells))\n            )\n    return tuple(components)\n\n\ndef _component_crop(\n    grid: Grid,\n    background: int,\n    ordering: str,\n    isolate: bool,\n) -> Grid | None:\n    components = _components(grid, background)\n    if not components:\n        return None\n    sort_keys = {\n        "largest": lambda obj: (-obj[1], obj[2], obj[3], obj[0]),\n        "smallest": lambda obj: (obj[1], obj[2], obj[3], obj[0]),\n        "topmost": lambda obj: (obj[2], obj[3], -obj[1], obj[0]),\n        "bottommost": lambda obj: (-obj[2], obj[3], -obj[1], obj[0]),\n        "leftmost": lambda obj: (obj[3], obj[2], -obj[1], obj[0]),\n        "rightmost": lambda obj: (-obj[3], obj[2], -obj[1], obj[0]),\n    }\n    selected = sorted(components, key=sort_keys[ordering])[0]\n    _, _, top, left, cells = selected\n    bottom = max(r for r, _ in cells) + 1\n    right = max(c for _, c in cells) + 1\n    if isolate:\n        return tuple(\n            tuple(grid[r][c] if (r, c) in cells else background for c in range(left, right))\n            for r in range(top, bottom)\n        )\n    return crop(grid, top, left, bottom, right)\n\n\ndef _learn_color_map(examples: list[tuple[Grid, Grid]]) -> dict[int, int] | None:\n    mapping: dict[int, int] = {}\n    for source, target in examples:\n        if shape(source) != shape(target):\n            return None\n        for srow, trow in zip(source, target):\n            for old, new in zip(srow, trow):\n                if old in mapping and mapping[old] != new:\n                    return None\n                mapping[old] = new\n    return mapping if any(old != new for old, new in mapping.items()) else None\n\n\ndef build_operations(examples: list[tuple[Grid, Grid]]) -> list[Operation]:\n    operations: list[Operation] = [\n        Operation("identity", lambda grid: grid, 0),\n        Operation("rotate_90", lambda grid: _rotate(grid, 1)),\n        Operation("rotate_180", lambda grid: _rotate(grid, 2)),\n        Operation("rotate_270", lambda grid: _rotate(grid, 3)),\n        Operation("flip_horizontal", _flip_horizontal),\n        Operation("flip_vertical", _flip_vertical),\n        Operation("transpose", lambda grid: tuple(tuple(row) for row in zip(*grid))),\n        Operation("anti_transpose", lambda grid: _rotate(_flip_horizontal(grid), 1)),\n    ]\n\n    color_map = _learn_color_map(examples)\n    if color_map is not None:\n        operations.append(\n            Operation(\n                "global_color_map",\n                lambda grid, mapping=color_map: tuple(\n                    tuple(mapping.get(cell, cell) for cell in row) for row in grid\n                ),\n                2,\n            )\n        )\n\n    for background in _background_candidates(examples):\n        operations.append(\n            Operation(\n                f"crop_background_{background}",\n                lambda grid, bg=background: (\n                    None\n                    if (box := bounding_box(grid, bg)) is None\n                    else crop(grid, *box)\n                ),\n                2,\n            )\n        )\n        for ordering in ("largest", "smallest", "topmost", "bottommost", "leftmost", "rightmost"):\n            for isolate in (False, True):\n                operations.append(\n                    Operation(\n                        f"crop_{ordering}_component_bg{background}_isolate{int(isolate)}",\n                        lambda grid, bg=background, order=ordering, only=isolate: _component_crop(\n                            grid, bg, order, only\n                        ),\n                        3,\n                    )\n                )\n\n    colors = sorted({cell for source, _ in examples for row in source for cell in row})\n    for color in colors:\n        operations.append(\n            Operation(\n                f"isolate_color_{color}",\n                lambda grid, selected=color: _isolate_color(grid, selected),\n                2,\n            )\n        )\n        operations.append(\n            Operation(\n                f"remove_color_{color}",\n                lambda grid, removed=color: _remove_color(grid, removed),\n                2,\n            )\n        )\n    for dx, dy in _infer_translations(examples):\n        operations.append(\n            Operation(\n                f"translate_x{dx}_y{dy}",\n                lambda grid, x=dx, y=dy: _translate(grid, x, y),\n                2,\n            )\n        )\n\n    for factor_y in range(2, 4):\n        for factor_x in range(2, 4):\n            operations.extend(\n                [\n                    Operation(\n                        f"tile_{factor_y}x{factor_x}",\n                        lambda grid, fy=factor_y, fx=factor_x: tuple(\n                            row\n                            for _ in range(fy)\n                            for source_row in grid\n                            for row in (source_row * fx,)\n                        ),\n                        3,\n                    ),\n                    Operation(\n                        f"scale_{factor_y}x{factor_x}",\n                        lambda grid, fy=factor_y, fx=factor_x: tuple(\n                            expanded_row\n                            for source_row in grid\n                            for expanded_row in (\n                                tuple(cell for cell in source_row for _ in range(fx)),\n                            )\n                            for _ in range(fy)\n                        ),\n                        3,\n                    ),\n                ]\n            )\n    from .operations_extra import build_extra_operations\n\n    operations.extend(build_extra_operations(examples))\n    return operations\n', 'operations_extra.py': '"""Additional exact transformations: symmetry, gravity, fills, grid splits and learned local rules."""\n\nfrom __future__ import annotations\n\nfrom collections import Counter, deque\nfrom typing import Callable\n\nfrom .grids import Grid, most_common_color, shape\nfrom .operations import Operation\n\n\ndef _hflip(grid: Grid) -> Grid:\n    return tuple(tuple(reversed(row)) for row in grid)\n\n\ndef _vflip(grid: Grid) -> Grid:\n    return tuple(reversed(grid))\n\n\ndef _hconcat(left: Grid, right: Grid) -> Grid:\n    return tuple(a + b for a, b in zip(left, right))\n\n\ndef _symmetry_ops() -> list[Operation]:\n    return [\n        Operation("mirror_right", lambda g: _hconcat(g, _hflip(g)), 3),\n        Operation("mirror_left", lambda g: _hconcat(_hflip(g), g), 3),\n        Operation("mirror_down", lambda g: g + _vflip(g), 3),\n        Operation("mirror_up", lambda g: _vflip(g) + g, 3),\n        Operation(\n            "mirror_quad",\n            lambda g: (lambda top: top + _vflip(top))(_hconcat(g, _hflip(g))),\n            3,\n        ),\n        Operation("keep_left_half", lambda g: tuple(r[: len(r) // 2] for r in g) if len(g[0]) > 1 else None, 3),\n        Operation("keep_right_half", lambda g: tuple(r[(len(r) + 1) // 2 :] for r in g) if len(g[0]) > 1 else None, 3),\n        Operation("keep_top_half", lambda g: g[: len(g) // 2] if len(g) > 1 else None, 3),\n        Operation("keep_bottom_half", lambda g: g[(len(g) + 1) // 2 :] if len(g) > 1 else None, 3),\n        Operation(\n            "complete_symmetry",\n            lambda g: _complete_symmetry(g),\n            3,\n        ),\n        Operation("self_tile", _self_tile, 3),\n        Operation("self_tile_inverse", lambda g: _self_tile(g, inverse=True), 3),\n    ]\n\n\ndef _self_tile(grid: Grid, inverse: bool = False) -> Grid | None:\n    height, width = shape(grid)\n    if height * height > 30 or width * width > 30:\n        return None\n    background = most_common_color(grid)\n    rows = []\n    for r in range(height):\n        for tr in range(height):\n            row: list[int] = []\n            for c in range(width):\n                use = (grid[r][c] != background) != inverse\n                row.extend(grid[tr] if use else (background,) * width)\n            rows.append(tuple(row))\n    return tuple(rows)\n\n\ndef _complete_symmetry(grid: Grid) -> Grid | None:\n    """Fill the most common colour cells from their mirror positions."""\n    height, width = shape(grid)\n    noise = most_common_color(grid)\n    counts = Counter(cell for row in grid for cell in row)\n    if len(counts) < 2:\n        return None\n    least = min(counts, key=lambda color: (counts[color], color))\n    rows = [list(row) for row in grid]\n    changed = False\n    for r in range(height):\n        for c in range(width):\n            if grid[r][c] != least:\n                continue\n            for mr, mc in ((r, width - 1 - c), (height - 1 - r, c), (height - 1 - r, width - 1 - c)):\n                if grid[mr][mc] != least:\n                    rows[r][c] = grid[mr][mc]\n                    changed = True\n                    break\n    del noise\n    return tuple(tuple(row) for row in rows) if changed else None\n\n\ndef _gravity(grid: Grid, direction: str) -> Grid:\n    background = most_common_color(grid)\n    height, width = shape(grid)\n    rows = [[background] * width for _ in range(height)]\n    if direction in ("down", "up"):\n        for c in range(width):\n            column = [grid[r][c] for r in range(height) if grid[r][c] != background]\n            if direction == "down":\n                for index, value in enumerate(reversed(column)):\n                    rows[height - 1 - index][c] = value\n            else:\n                for index, value in enumerate(column):\n                    rows[index][c] = value\n    else:\n        for r in range(height):\n            line = [cell for cell in grid[r] if cell != background]\n            if direction == "right":\n                for index, value in enumerate(reversed(line)):\n                    rows[r][width - 1 - index] = value\n            else:\n                for index, value in enumerate(line):\n                    rows[r][index] = value\n    return tuple(tuple(row) for row in rows)\n\n\ndef _enclosed_cells(grid: Grid, background: int) -> set[tuple[int, int]]:\n    height, width = shape(grid)\n    seen: set[tuple[int, int]] = set()\n    queue: deque[tuple[int, int]] = deque()\n    for r in range(height):\n        for c in range(width):\n            if (r in (0, height - 1) or c in (0, width - 1)) and grid[r][c] == background:\n                seen.add((r, c))\n                queue.append((r, c))\n    while queue:\n        r, c = queue.popleft()\n        for nr, nc in ((r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)):\n            if 0 <= nr < height and 0 <= nc < width and (nr, nc) not in seen and grid[nr][nc] == background:\n                seen.add((nr, nc))\n                queue.append((nr, nc))\n    return {\n        (r, c)\n        for r in range(height)\n        for c in range(width)\n        if grid[r][c] == background and (r, c) not in seen\n    }\n\n\ndef _fill_enclosed(grid: Grid, fill: int) -> Grid | None:\n    background = most_common_color(grid)\n    cells = _enclosed_cells(grid, background)\n    if not cells:\n        return None\n    return tuple(\n        tuple(fill if (r, c) in cells else cell for c, cell in enumerate(row))\n        for r, row in enumerate(grid)\n    )\n\n\ndef _split_parts(grid: Grid, axis: str, parts: int) -> list[Grid] | None:\n    height, width = shape(grid)\n    size = height if axis == "v" else width\n    if axis == "v":\n        uniform = {r for r in range(height) if len(set(grid[r])) == 1}\n        colors = {grid[r][0] for r in uniform}\n    else:\n        uniform = {c for c in range(width) if len({grid[r][c] for r in range(height)}) == 1}\n        colors = {grid[0][c] for c in uniform}\n    bounds: list[tuple[int, int]] | None = None\n    if (size - (parts - 1)) % parts == 0 and size > parts:\n        step = (size - (parts - 1)) // parts\n        lines = [(i + 1) * step + i for i in range(parts - 1)]\n        if all(line in uniform for line in lines) and len({_line_color(grid, axis, l) for l in lines}) == 1:\n            bounds = [(i * (step + 1), i * (step + 1) + step) for i in range(parts)]\n    if bounds is None:\n        del colors\n        if size % parts:\n            return None\n        step = size // parts\n        bounds = [(i * step, (i + 1) * step) for i in range(parts)]\n    pieces: list[Grid] = []\n    for start, end in bounds:\n        if end <= start:\n            return None\n        pieces.append(grid[start:end] if axis == "v" else tuple(row[start:end] for row in grid))\n    return pieces\n\n\ndef _line_color(grid: Grid, axis: str, index: int) -> int:\n    return grid[index][0] if axis == "v" else grid[0][index]\n\n\ndef _learn_split_rule(\n    examples: list[tuple[Grid, Grid]], axis: str, parts: int\n) -> Callable[[Grid], Grid | None] | None:\n    table: dict[tuple[int, ...], int] = {}\n    backgrounds: set[int] = set()\n    for source, target in examples:\n        pieces = _split_parts(source, axis, parts)\n        if pieces is None or shape(pieces[0]) != shape(target):\n            return None\n        for piece in pieces:\n            backgrounds.add(most_common_color(source))\n        height, width = shape(target)\n        for r in range(height):\n            for c in range(width):\n                key = tuple(int(piece[r][c] != 0) for piece in pieces)\n                value = target[r][c]\n                if table.setdefault(key, value) != value:\n                    return None\n    if len(table) < 2:\n        return None\n\n    def apply(grid: Grid) -> Grid | None:\n        pieces = _split_parts(grid, axis, parts)\n        if pieces is None:\n            return None\n        height, width = shape(pieces[0])\n        rows = []\n        for r in range(height):\n            row = []\n            for c in range(width):\n                key = tuple(int(piece[r][c] != 0) for piece in pieces)\n                if key not in table:\n                    return None\n                row.append(table[key])\n            rows.append(tuple(row))\n        return tuple(rows)\n\n    return apply\n\n\ndef _patch(grid: Grid, r: int, c: int, kind: str) -> tuple[int, ...]:\n    height, width = shape(grid)\n\n    def at(rr: int, cc: int) -> int:\n        return grid[rr][cc] if 0 <= rr < height and 0 <= cc < width else -1\n\n    if kind == "cross":\n        return (at(r, c), at(r - 1, c), at(r + 1, c), at(r, c - 1), at(r, c + 1))\n    if kind == "square":\n        return tuple(at(r + dr, c + dc) for dr in (-1, 0, 1) for dc in (-1, 0, 1))\n    if kind == "row_col":\n        row_colors = tuple(sorted(set(grid[r]) - {most_common_color(grid)}))\n        col_colors = tuple(sorted({grid[i][c] for i in range(height)} - {most_common_color(grid)}))\n        return (at(r, c),) + row_colors + (-2,) + col_colors\n    return (at(r, c),)\n\n\ndef _learn_local_rule(\n    examples: list[tuple[Grid, Grid]], kind: str\n) -> Callable[[Grid], Grid | None] | None:\n    table: dict[tuple[int, ...], int] = {}\n    for source, target in examples:\n        if shape(source) != shape(target):\n            return None\n        for r, row in enumerate(source):\n            for c in range(len(row)):\n                key = _patch(source, r, c, kind)\n                value = target[r][c]\n                if table.setdefault(key, value) != value:\n                    return None\n    if all(key[0] == value for key, value in table.items()):\n        return None\n\n    def apply(grid: Grid) -> Grid | None:\n        rows = []\n        for r, row in enumerate(grid):\n            new_row = []\n            for c in range(len(row)):\n                key = _patch(grid, r, c, kind)\n                if key not in table:\n                    return None\n                new_row.append(table[key])\n            rows.append(tuple(new_row))\n        return tuple(rows)\n\n    return apply\n\n\ndef _downscale(grid: Grid, fy: int, fx: int) -> Grid | None:\n    height, width = shape(grid)\n    if height % fy or width % fx:\n        return None\n    return tuple(tuple(grid[r][c] for c in range(0, width, fx)) for r in range(0, height, fy))\n\n\ndef _remove_noise(grid: Grid) -> Grid | None:\n    background = most_common_color(grid)\n    height, width = shape(grid)\n    rows = [list(row) for row in grid]\n    changed = False\n    for r in range(height):\n        for c in range(width):\n            color = grid[r][c]\n            if color == background:\n                continue\n            neighbours = [\n                grid[nr][nc]\n                for nr, nc in ((r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1))\n                if 0 <= nr < height and 0 <= nc < width\n            ]\n            if color not in neighbours:\n                rows[r][c] = background\n                changed = True\n    return tuple(tuple(row) for row in rows) if changed else None\n\n\ndef build_extra_operations(examples: list[tuple[Grid, Grid]]) -> list[Operation]:\n    operations = _symmetry_ops()\n    for direction in ("down", "up", "left", "right"):\n        operations.append(Operation(f"gravity_{direction}", lambda g, d=direction: _gravity(g, d), 3))\n    operations.append(Operation("remove_noise", _remove_noise, 3))\n    for factor_y in range(1, 4):\n        for factor_x in range(1, 4):\n            if (factor_y, factor_x) != (1, 1):\n                operations.append(\n                    Operation(\n                        f"downscale_{factor_y}x{factor_x}",\n                        lambda g, fy=factor_y, fx=factor_x: _downscale(g, fy, fx),\n                        3,\n                    )\n                )\n\n    output_colors = sorted({cell for _, target in examples for row in target for cell in row})\n    for color in output_colors:\n        operations.append(\n            Operation(f"fill_enclosed_{color}", lambda g, fill=color: _fill_enclosed(g, fill), 3)\n        )\n\n    for axis in ("v", "h"):\n        for parts in (2, 3, 4):\n            rule = _learn_split_rule(examples, axis, parts)\n            if rule is not None:\n                operations.append(Operation(f"split_{axis}{parts}_boolean_rule", rule, 5))\n\n    for kind in ("cross", "square", "row_col", "cell"):\n        rule = _learn_local_rule(examples, kind)\n        if rule is not None:\n            operations.append(Operation(f"local_rule_{kind}", rule, 5))\n    return [Operation(o.name, o.apply, max(o.cost, 4)) for o in operations]\n', 'proposer.py': '"""Structured program proposals and validation-safe prompts."""\n\nfrom __future__ import annotations\n\nimport json\nimport re\nfrom dataclasses import dataclass, replace\nfrom typing import Any, Protocol\n\nfrom .grids import Grid, as_grid, to_json_grid\nfrom .operations import Operation\n\n\n@dataclass(frozen=True)\nclass ProgramProposal:\n    operation_names: tuple[str, ...] | None\n    rationale: str = ""\n    error: str | None = None\n    calibration_output: Grid | None = None\n    direct_test_outputs: tuple[Grid, ...] | None = None\n    direct_output_error: str | None = None\n\n\nclass ProgramProposer(Protocol):\n    def propose(\n        self,\n        task: dict[str, Any],\n        operations: list[Operation],\n        count: int,\n    ) -> list[ProgramProposal]: ...\n\n\ndef describe_operation(name: str) -> str:\n    descriptions = {\n        "identity": "leave the grid unchanged",\n        "rotate_90": "rotate the whole grid 90 degrees clockwise",\n        "rotate_180": "rotate the whole grid 180 degrees",\n        "rotate_270": "rotate the whole grid 90 degrees counterclockwise",\n        "flip_horizontal": "mirror the whole grid left to right",\n        "flip_vertical": "mirror the whole grid top to bottom",\n        "transpose": "transpose rows and columns",\n        "anti_transpose": "reflect across the top-right to bottom-left diagonal",\n        "global_color_map": "apply one globally consistent color remapping inferred from the examples",\n    }\n    if name in descriptions:\n        return descriptions[name]\n    match = re.fullmatch(r"crop_background_(\\d+)", name)\n    if match:\n        return f"crop to the tight bounding box of cells not color {match.group(1)}"\n    match = re.fullmatch(r"crop_excluding_color_(\\d+)", name)\n    if match:\n        return f"crop to the tight bounding box of cells not color {match.group(1)}"\n    match = re.fullmatch(\n        r"crop_(largest|smallest|topmost|bottommost|leftmost|rightmost)_component_bg(\\d+)_isolate([01])",\n        name,\n    )\n    if match:\n        selected, background, isolate = match.groups()\n        detail = "keep only that component" if isolate == "1" else "keep the selected crop contents"\n        return (\n            f"select the {selected} connected component excluding background color "\n            f"{background}, crop to its bounds, and {detail}"\n        )\n    match = re.fullmatch(r"(isolate|remove)_color_(\\d+)", name)\n    if match:\n        action, color = match.groups()\n        if action == "isolate":\n            return f"keep color {color} and replace all other cells with the grid\'s modal color"\n        return f"replace color {color} with the grid\'s modal color"\n    match = re.fullmatch(r"translate_x(-?\\d+)_y(-?\\d+)", name)\n    if match:\n        dx, dy = match.groups()\n        return f"shift all cells by {dx} columns and {dy} rows, filling exposed cells with the modal color"\n    match = re.fullmatch(r"(tile|scale)_(\\d+)x(\\d+)", name)\n    if match:\n        operation, height, width = match.groups()\n        action = "repeat the full grid" if operation == "tile" else "repeat each cell"\n        return f"{action} {height} times vertically and {width} times horizontally"\n    return f"apply the predefined operation named {name}"\n\n\ndef build_program_prompt(\n    task: dict[str, Any],\n    operations: list[Operation],\n    candidate_count: int = 3,\n    calibration_input: Grid | None = None,\n) -> str:\n    if candidate_count < 1:\n        raise ValueError("At least one candidate program is required.")\n    training_examples = [\n        {\n            "input": to_json_grid(pair["input"]),\n            "output": to_json_grid(pair["output"]),\n        }\n        for pair in task["train"]\n    ]\n    test_inputs = [{"input": to_json_grid(pair["input"])} for pair in task["test"]]\n    operation_catalog = [\n        {"name": operation.name, "meaning": describe_operation(operation.name)}\n        for operation in operations\n    ]\n    calibration_section = ""\n    response_fields = ""\n    program_schema = ",".join(\n        \'["operation_name"]\' for _ in range(candidate_count)\n    )\n    if calibration_input is not None:\n        calibration_section = (\n            f"CALIBRATION_INPUT={json.dumps(to_json_grid(calibration_input), separators=(\',\', \':\'))}\\n"\n            "Predict this held-out training example using only TRAINING_EXAMPLES, "\n            "then predict TEST_INPUTS. Write each output as a list of digit strings, "\n            "one string per row. The solver will use the calibration prediction only "\n            "if it exactly matches the withheld training output.\\n"\n        )\n        response_fields = (\n            \',"calibration_output":["012","345"],\'\n            \'"test_outputs":[["012","345"]]\'\n        )\n    return (\n        "Infer the single transformation rule demonstrated by these ARC examples. "\n        "Choose a sequence of 1 to 5 operations only from the allowed catalog. "\n        "The sequence is applied left to right to each test input. Do not write or "\n        "execute code, do not invent operations, and do not use outside information. "\n        "Prefer the shortest rule that explains every demonstration. Test inputs "\n        "are included only to clarify which grids the rule will be applied to. "\n        f"Propose exactly {candidate_count} distinct candidate programs. "\n        "Return exactly one JSON object and no other text. Do not include "\n        "rationales. Use only operation names from the catalog. "\n        f\'Output schema: {{"programs":[{program_schema}]{response_fields}}}.\\n\\n\'\n        f"TRAINING_EXAMPLES={json.dumps(training_examples, separators=(\',\', \':\'))}\\n"\n        f"{calibration_section}"\n        f"TEST_INPUTS={json.dumps(test_inputs, separators=(\',\', \':\'))}\\n"\n        f"ALLOWED_OPERATIONS={json.dumps(operation_catalog, separators=(\',\', \':\'))}"\n    )\n\n\ndef _find_program_object(response: str) -> dict[str, Any] | None:\n    decoder = json.JSONDecoder()\n    parsed: dict[str, Any] | None = None\n    for position, character in enumerate(response):\n        if character != "{":\n            continue\n        try:\n            value, _ = decoder.raw_decode(response, position)\n        except json.JSONDecodeError:\n            continue\n        if isinstance(value, dict) and (\n            "programs" in value or "program" in value\n        ):\n            parsed = value\n    return parsed\n\n\ndef _find_json_field(response: str, field: str) -> Any | None:\n    decoder = json.JSONDecoder()\n    pattern = re.compile(rf\'"{re.escape(field)}"\\s*:\')\n    value: Any | None = None\n    for match in pattern.finditer(response):\n        position = match.end()\n        while position < len(response) and response[position].isspace():\n            position += 1\n        try:\n            value, _ = decoder.raw_decode(response, position)\n        except json.JSONDecodeError:\n            continue\n    return value\n\n\ndef _parse_program_fields(\n    raw_program: Any,\n    rationale: Any,\n    allowed_operation_names: set[str],\n    max_length: int,\n) -> ProgramProposal:\n    if not isinstance(raw_program, list) or not raw_program:\n        return ProgramProposal(\n            None,\n            error="The program must be a non-empty list of operation names.",\n        )\n    if len(raw_program) > max_length:\n        return ProgramProposal(\n            None,\n            error=f"The program exceeds the {max_length}-operation limit.",\n        )\n    if any(not isinstance(name, str) for name in raw_program):\n        return ProgramProposal(\n            None,\n            error="Every program operation must be a string.",\n        )\n    names = tuple(raw_program)\n    unknown = sorted(set(names) - allowed_operation_names)\n    if unknown:\n        return ProgramProposal(\n            None,\n            error=f"Unknown operation name(s): {\', \'.join(unknown)}.",\n        )\n    if not isinstance(rationale, str):\n        return ProgramProposal(None, error="The rationale field must be a string.")\n    return ProgramProposal(names, rationale)\n\n\ndef _parse_output_grid(value: Any) -> Grid:\n    if isinstance(value, list) and value and all(isinstance(row, str) for row in value):\n        if any(not row or any(character not in "0123456789" for character in row) for row in value):\n            raise ValueError("Grid rows must be non-empty strings of ARC color digits.")\n        value = [[int(character) for character in row] for row in value]\n    return as_grid(value)\n\n\ndef parse_program_proposals(\n    response: str,\n    allowed_operation_names: set[str],\n    count: int,\n    max_length: int = 5,\n    test_count: int = 0,\n    require_calibration: bool = False,\n) -> list[ProgramProposal]:\n    if count < 1:\n        raise ValueError("At least one candidate program is required.")\n    parsed = _find_program_object(response)\n    if parsed is None:\n        raw_programs = _find_json_field(response, "programs")\n        raw_program = _find_json_field(response, "program")\n        if raw_programs is None and raw_program is None:\n            return [\n                ProgramProposal(\n                    None,\n                    error="Response did not contain a JSON program object.",\n                )\n                for _ in range(count)\n            ]\n        parsed = {}\n        if raw_programs is not None:\n            parsed["programs"] = raw_programs\n        if raw_program is not None:\n            parsed["program"] = raw_program\n        calibration_output = _find_json_field(response, "calibration_output")\n        test_outputs = _find_json_field(response, "test_outputs")\n        if calibration_output is not None:\n            parsed["calibration_output"] = calibration_output\n        if test_outputs is not None:\n            parsed["test_outputs"] = test_outputs\n\n    raw_programs = parsed.get("programs")\n    if raw_programs is None:\n        raw_programs = [parsed.get("program")]\n    if not isinstance(raw_programs, list):\n        raw_programs = []\n\n    proposals: list[ProgramProposal] = []\n    for raw_program in raw_programs[:count]:\n        if isinstance(raw_program, dict):\n            proposals.append(\n                _parse_program_fields(\n                    raw_program.get("program"),\n                    raw_program.get("rationale", ""),\n                    allowed_operation_names,\n                    max_length,\n                )\n            )\n        else:\n            proposals.append(\n                _parse_program_fields(\n                    raw_program,\n                    "",\n                    allowed_operation_names,\n                    max_length,\n                )\n            )\n\n    while len(proposals) < count:\n        proposals.append(\n            ProgramProposal(\n                None,\n                error=(\n                    f"Response returned {len(raw_programs)} programs; "\n                    f"{count} were requested."\n                ),\n            )\n        )\n\n    if require_calibration:\n        direct_output_error: str | None = None\n        calibration_output: Grid | None = None\n        direct_test_outputs: tuple[Grid, ...] | None = None\n        try:\n            if "calibration_output" not in parsed:\n                raise ValueError("Response did not include calibration_output.")\n            calibration_output = _parse_output_grid(parsed["calibration_output"])\n            raw_test_outputs = parsed.get("test_outputs")\n            if not isinstance(raw_test_outputs, list):\n                raise ValueError("Response did not include a test_outputs list.")\n            if len(raw_test_outputs) != test_count:\n                raise ValueError(\n                    f"Expected {test_count} test outputs, got {len(raw_test_outputs)}."\n                )\n            direct_test_outputs = tuple(\n                _parse_output_grid(output) for output in raw_test_outputs\n            )\n        except ValueError as error:\n            direct_output_error = str(error)\n        if proposals:\n            proposals[0] = replace(\n                proposals[0],\n                calibration_output=calibration_output,\n                direct_test_outputs=direct_test_outputs,\n                direct_output_error=direct_output_error,\n            )\n    return proposals\n\n\ndef parse_program_proposal(\n    response: str,\n    allowed_operation_names: set[str],\n    max_length: int = 5,\n) -> ProgramProposal:\n    return parse_program_proposals(\n        response,\n        allowed_operation_names,\n        count=1,\n        max_length=max_length,\n    )[0]\n', 'local_model.py': '"""Offline Qwen program proposer for use with attached Kaggle model weights."""\n\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\nfrom typing import Any\n\nfrom .grids import Grid, as_grid\nfrom .operations import Operation, build_operations\nfrom .proposer import ProgramProposal, build_program_prompt, parse_program_proposals\n\n\nMODEL_NAME = "Qwen/Qwen3-30B-A3B-Instruct-2507"\nMIN_TRANSFORMERS_VERSION = (4, 51)\n\n\ndef find_local_qwen_model(input_root: Path) -> Path:\n    candidates: set[Path] = set()\n    for config_path in input_root.rglob("config.json"):\n        with config_path.open(encoding="utf-8") as stream:\n            config = json.load(stream)\n        if config.get("model_type") != "qwen3_moe":\n            continue\n        if not any(\n            list(config_path.parent.glob(pattern))\n            for pattern in ("*.safetensors", "*.bin", "*.safetensors.index.json")\n        ):\n            continue\n        candidates.add(config_path.parent.resolve())\n    if not candidates:\n        raise FileNotFoundError(\n            f"No local Qwen3-MoE model with weights was found under {input_root}. "\n            f"Attach {MODEL_NAME} as a Kaggle Model input; runtime downloads are disabled."\n        )\n    if len(candidates) != 1:\n        paths = ", ".join(str(path) for path in sorted(candidates))\n        raise ValueError(f"Expected one attached Qwen3-MoE model, found: {paths}")\n    return next(iter(candidates))\n\n\nclass LocalQwenProgramProposer:\n    def __init__(\n        self,\n        tokenizer: Any,\n        model: Any,\n        max_new_tokens: int = 2048,\n        seed: int = 2026,\n    ) -> None:\n        if max_new_tokens < 64:\n            raise ValueError("max_new_tokens must be at least 64.")\n        self.tokenizer = tokenizer\n        self.model = model\n        self.max_new_tokens = max_new_tokens\n        self.seed = seed\n\n    @classmethod\n    def from_pretrained(cls, model_path: Path) -> LocalQwenProgramProposer:\n        import torch\n        import transformers\n        from transformers import AutoModelForCausalLM, AutoTokenizer\n\n        version_parts = tuple(\n            int(part) for part in transformers.__version__.split(".")[:2]\n        )\n        if version_parts < MIN_TRANSFORMERS_VERSION:\n            raise RuntimeError(\n                "Qwen3-MoE requires transformers>=4.51.0; "\n                f"the Kaggle image has {transformers.__version__}."\n            )\n        if not torch.cuda.is_available():\n            raise RuntimeError("The attached Qwen model requires a CUDA GPU.")\n\n        tokenizer = AutoTokenizer.from_pretrained(\n            str(model_path),\n            local_files_only=True,\n            trust_remote_code=False,\n        )\n        model = AutoModelForCausalLM.from_pretrained(\n            str(model_path),\n            dtype=torch.bfloat16,\n            device_map="auto",\n            low_cpu_mem_usage=True,\n            local_files_only=True,\n            trust_remote_code=False,\n        )\n        model.eval()\n        return cls(tokenizer, model)\n\n    def propose(\n        self,\n        task: dict[str, Any],\n        operations: list[Operation],\n        count: int,\n    ) -> list[ProgramProposal]:\n        if count < 1:\n            raise ValueError("At least one model proposal is required.")\n        train = task["train"]\n        has_calibration = len(train) >= 2\n        prompt_task = dict(task)\n        calibration_input: Grid | None = None\n        prompt_operations = operations\n        if has_calibration:\n            calibration_pair = train[-1]\n            prompt_task["train"] = train[:-1]\n            calibration_input = as_grid(calibration_pair["input"])\n            visible_examples = [\n                (\n                    as_grid(pair["input"]),\n                    as_grid(pair["output"]),\n                )\n                for pair in prompt_task["train"]\n            ]\n            prompt_operations = build_operations(visible_examples)\n        operation_names = {operation.name for operation in prompt_operations}\n        prompt = build_program_prompt(\n            prompt_task,\n            prompt_operations,\n            candidate_count=count,\n            calibration_input=calibration_input,\n        )\n        prompt_text = self.tokenizer.apply_chat_template(\n            [{"role": "user", "content": prompt}],\n            tokenize=False,\n            add_generation_prompt=True,\n        )\n        model_inputs = self.tokenizer(\n            prompt_text,\n            return_tensors="pt",\n            add_special_tokens=False,\n        )\n        input_ids = model_inputs["input_ids"]\n        device = self.model.get_input_embeddings().weight.device\n        input_ids = input_ids.to(device)\n        attention_mask = model_inputs.get("attention_mask")\n        if attention_mask is not None:\n            attention_mask = attention_mask.to(device)\n        import torch\n\n        torch.manual_seed(self.seed)\n        if torch.cuda.is_available():\n            torch.cuda.manual_seed_all(self.seed)\n        output_ids = self.model.generate(\n            input_ids,\n            attention_mask=attention_mask,\n            max_new_tokens=self.max_new_tokens,\n            do_sample=True,\n            temperature=0.7,\n            top_p=0.9,\n            pad_token_id=self.tokenizer.eos_token_id,\n        )\n        response = self.tokenizer.decode(\n            output_ids[0, input_ids.shape[1] :],\n            skip_special_tokens=True,\n        )\n        return parse_program_proposals(\n            response,\n            operation_names,\n            count,\n            test_count=len(task["test"]),\n            require_calibration=has_calibration,\n        )\n', 'inference.py': '"""Task-level program synthesis and prediction ranking."""\n\nfrom __future__ import annotations\n\nfrom collections import defaultdict\nfrom dataclasses import dataclass\nfrom typing import Any\n\nfrom .grids import Grid, as_grid\nfrom .operations import Operation, build_operations\nfrom .proposer import ProgramProposer\n\n_SYNTHESIS_BEAM_WIDTH = 128\n_PATHS_PER_STATE = 3\n_EXTRA_OPERATION_COST = 4\n_LEARNED_PREFIXES = ("local_rule_", "split_")\n\n\n@dataclass(frozen=True)\nclass Program:\n    operations: tuple[Operation, ...]\n\n    @property\n    def cost(self) -> int:\n        return sum(operation.cost for operation in self.operations)\n\n    @property\n    def name(self) -> str:\n        return " -> ".join(operation.name for operation in self.operations)\n\n    def apply(self, grid: Grid) -> Grid | None:\n        result: Grid | None = grid\n        for operation in self.operations:\n            if result is None:\n                return None\n            result = operation.apply(result)\n            if result is not None and (\n                not result\n                or not result[0]\n                or len(result) > 30\n                or len(result[0]) > 30\n                or any(len(row) != len(result[0]) for row in result)\n                or any(cell not in range(10) for row in result for cell in row)\n            ):\n                return None\n        return result\n\n\ndef _program_signature(program: Program) -> tuple[str, ...]:\n    return tuple(operation.name for operation in program.operations)\n\n\n@dataclass(frozen=True)\nclass SolveResult:\n    attempt_1: Grid\n    attempt_2: Grid\n    program_names: tuple[str, ...]\n    consistent_program_count: int\n    model_proposal_count: int = 0\n    model_accepted_count: int = 0\n    model_rejected_count: int = 0\n    model_rejection_reasons: tuple[str, ...] = ()\n    model_calibration_passed: bool = False\n    model_direct_output_error: str | None = None\n\n\ndef _beam_candidate_programs(\n    examples: list[tuple[Grid, Grid]],\n    max_depth: int,\n    operations: list[Operation],\n) -> list[Program]:\n    sources = tuple(source for source, _ in examples)\n    targets = tuple(target for _, target in examples)\n    frontier: dict[tuple[Grid, ...], list[tuple[Operation, ...]]] = {\n        sources: [()]\n    }\n    visited_depth = {sources: 0}\n\n    def distance(state: tuple[Grid, ...]) -> int:\n        total = 0\n        for grid, target in zip(state, targets):\n            height, width = len(grid), len(grid[0])\n            target_height, target_width = len(target), len(target[0])\n            total += 1000 * (\n                abs(height - target_height) + abs(width - target_width)\n            )\n            total += sum(\n                grid[row][column] != target[row][column]\n                for row in range(min(height, target_height))\n                for column in range(min(width, target_width))\n            )\n        return total\n\n    def valid_grid(grid: Grid | None) -> bool:\n        return (\n            grid is not None\n            and 0 < len(grid) <= 30\n            and 0 < len(grid[0]) <= 30\n            and all(len(row) == len(grid[0]) for row in grid)\n            and all(cell in range(10) for row in grid for cell in row)\n        )\n\n    for depth in range(1, max_depth + 1):\n        next_paths: dict[tuple[Grid, ...], list[tuple[Operation, ...]]] = defaultdict(list)\n        exact_programs: dict[tuple[str, ...], Program] = {}\n        for state, prefixes in frontier.items():\n            for operation in operations:\n                if operation.name == "identity":\n                    continue\n                transformed_grids = tuple(operation.apply(grid) for grid in state)\n                if not all(valid_grid(grid) for grid in transformed_grids):\n                    continue\n                transformed = tuple(\n                    grid for grid in transformed_grids if grid is not None\n                )\n                if transformed == state:\n                    continue\n\n                for prefix in prefixes:\n                    sequence = prefix + (operation,)\n                    if transformed == targets:\n                        program = Program(sequence)\n                        exact_programs[_program_signature(program)] = program\n                        continue\n\n                    previous_depth = visited_depth.get(transformed)\n                    if previous_depth is not None and previous_depth < depth:\n                        continue\n                    visited_depth[transformed] = depth\n                    paths = next_paths[transformed]\n                    signature = tuple(item.name for item in sequence)\n                    if any(\n                        tuple(item.name for item in existing) == signature\n                        for existing in paths\n                    ):\n                        continue\n                    paths.append(sequence)\n\n        if exact_programs:\n            return sorted(\n                exact_programs.values(),\n                key=lambda program: (program.cost, program.name),\n            )\n        if depth == max_depth:\n            break\n\n        ranked_states = sorted(\n            next_paths.items(),\n            key=lambda item: (\n                distance(item[0]),\n                min(sum(operation.cost for operation in path) for path in item[1]),\n                min(" -> ".join(operation.name for operation in path) for path in item[1]),\n            ),\n        )[:_SYNTHESIS_BEAM_WIDTH]\n        frontier = {}\n        for state, paths in ranked_states:\n            frontier[state] = sorted(\n                paths,\n                key=lambda path: (\n                    sum(operation.cost for operation in path),\n                    " -> ".join(operation.name for operation in path),\n                ),\n            )[:_PATHS_PER_STATE]\n\n    return []\n\n\ndef _candidate_programs(\n    examples: list[tuple[Grid, Grid]],\n    max_depth: int,\n    operations: list[Operation],\n) -> list[Program]:\n    first_input, first_output = examples[0]\n    one_step: list[tuple[Operation, Grid | None]] = [\n        (operation, operation.apply(first_input)) for operation in operations\n    ]\n    exact_programs: list[Program] = [\n        Program((operation,))\n        for operation, result in one_step\n        if result == first_output\n    ]\n\n    if max_depth >= 2:\n        intermediates: dict[Grid, list[Operation]] = defaultdict(list)\n        for operation, result in one_step:\n            if (\n                result is not None\n                and result != first_output\n                and operation.name != "identity"\n            ):\n                intermediates[result].append(operation)\n        for intermediate, prefixes in intermediates.items():\n            for suffix in operations:\n                if suffix.name == "identity":\n                    continue\n                if suffix.apply(intermediate) == first_output:\n                    exact_programs.extend(\n                        Program((prefix, suffix)) for prefix in prefixes\n                    )\n\n    consistent: dict[tuple[str, ...], Program] = {}\n    for program in exact_programs:\n        if all(program.apply(source) == target for source, target in examples):\n            consistent[_program_signature(program)] = program\n    ranked = sorted(\n        consistent.values(),\n        key=lambda program: (program.cost, program.name),\n    )\n    if max_depth == 3 and not any(\n        not any(\n            operation.name.startswith(_LEARNED_PREFIXES)\n            for operation in program.operations\n        )\n        for program in ranked\n    ):\n        base_operations = [\n            operation for operation in operations if operation.cost < _EXTRA_OPERATION_COST\n        ]\n        return ranked + _beam_candidate_programs(examples, max_depth, base_operations)\n    return ranked\n\n\ndef solve_task(\n    task: dict[str, Any],\n    max_depth: int = 2,\n    proposer: ProgramProposer | None = None,\n    proposal_count: int = 3,\n) -> list[SolveResult]:\n    """Infer a transformation from all demonstrations and predict each test input."""\n    if max_depth not in (1, 2, 3):\n        raise ValueError("max_depth must be 1, 2, or 3.")\n    if proposal_count < 1:\n        raise ValueError("proposal_count must be positive.")\n    raw_train = task.get("train")\n    raw_test = task.get("test")\n    if not isinstance(raw_train, list) or not raw_train:\n        raise ValueError("Task must contain at least one training pair.")\n    if not isinstance(raw_test, list) or not raw_test:\n        raise ValueError("Task must contain at least one test input.")\n\n    examples: list[tuple[Grid, Grid]] = []\n    for pair in raw_train:\n        examples.append((as_grid(pair["input"]), as_grid(pair["output"])))\n    operations = build_operations(examples)\n    programs = _candidate_programs(examples, max_depth, operations)\n    test_inputs = [as_grid(pair["input"]) for pair in raw_test]\n    model_proposal_count = 0\n    model_accepted_count = 0\n    model_rejection_reasons: list[str] = []\n    model_program_signatures: set[tuple[str, ...]] = set()\n    model_direct_outputs: tuple[Grid, ...] | None = None\n    model_calibration_passed = False\n    model_direct_output_error: str | None = None\n    if proposer is not None:\n        safe_task = {\n            "train": [\n                {"input": source, "output": target}\n                for source, target in examples\n            ],\n            "test": [{"input": grid} for grid in test_inputs],\n        }\n        proposals = proposer.propose(safe_task, operations, proposal_count)\n        model_proposal_count = len(proposals)\n        operations_by_name = {operation.name: operation for operation in operations}\n        model_programs: dict[tuple[str, ...], Program] = {}\n        for proposal in proposals:\n            if proposal.direct_output_error is not None:\n                model_direct_output_error = proposal.direct_output_error\n            if (\n                proposal.calibration_output is not None\n                and proposal.direct_test_outputs is not None\n            ):\n                if proposal.calibration_output == examples[-1][1]:\n                    model_direct_outputs = proposal.direct_test_outputs\n                    model_calibration_passed = True\n            if proposal.error is not None:\n                model_rejection_reasons.append(proposal.error)\n                continue\n            if proposal.operation_names is None:\n                model_rejection_reasons.append("Proposal contained no operation sequence.")\n                continue\n            if not proposal.operation_names or len(proposal.operation_names) > 5:\n                model_rejection_reasons.append(\n                    "Program must contain between one and five operations."\n                )\n                continue\n            if any(not isinstance(name, str) for name in proposal.operation_names):\n                model_rejection_reasons.append(\n                    "Every proposed operation name must be a string."\n                )\n                continue\n            selected_operations = [\n                operations_by_name.get(name) for name in proposal.operation_names\n            ]\n            if any(operation is None for operation in selected_operations):\n                model_rejection_reasons.append("Proposal referenced an unavailable operation.")\n                continue\n            program = Program(\n                tuple(\n                    operation\n                    for operation in selected_operations\n                    if operation is not None\n                )\n            )\n            if not all(program.apply(source) == target for source, target in examples):\n                model_rejection_reasons.append(\n                    f"Program failed exact demonstration verification: {program.name}"\n                )\n                continue\n            signature = _program_signature(program)\n            model_programs[signature] = program\n            model_program_signatures.add(signature)\n        model_accepted_count = len(model_programs)\n        programs_by_signature = {\n            _program_signature(program): program\n            for program in programs + list(model_programs.values())\n        }\n        programs = sorted(\n            programs_by_signature.values(),\n            key=lambda program: (program.cost, program.name),\n        )\n    results: list[SolveResult] = []\n    for test_input in test_inputs:\n        votes: dict[Grid, list[Program]] = defaultdict(list)\n        for program in programs:\n            prediction = program.apply(test_input)\n            if prediction is not None:\n                votes[prediction].append(program)\n        ranked = sorted(\n            votes.items(),\n            key=lambda item: (\n                -len(\n                    {\n                        _program_signature(program)\n                        for program in item[1]\n                        if _program_signature(program) in model_program_signatures\n                    }\n                ),\n                -len(\n                    {\n                        program.operations[0].name\n                        for program in item[1]\n                        if program.operations[0].cost < _EXTRA_OPERATION_COST\n                    }\n                ),\n                min(program.cost for program in item[1]),\n                min(program.name for program in item[1]),\n            ),\n        )\n        matching_examples = {\n            target\n            for source, target in examples\n            if source == test_input\n        }\n        if len(matching_examples) == 1:\n            exact_example = next(iter(matching_examples))\n            results.append(\n                SolveResult(\n                    exact_example,\n                    exact_example,\n                    ("exact_demonstration_match",),\n                    len(programs),\n                    model_proposal_count,\n                    model_accepted_count,\n                    len(model_rejection_reasons),\n                    tuple(model_rejection_reasons),\n                    model_calibration_passed,\n                    model_direct_output_error,\n                )\n            )\n            continue\n        if ranked:\n            symbolic_first = ranked[0][0]\n            symbolic_second = ranked[1][0] if len(ranked) > 1 else symbolic_first\n            names = tuple(\n                min((program.name for program in group), key=lambda name: (len(name), name))\n                for _, group in ranked[:2]\n            )\n            if model_direct_outputs is not None:\n                first = model_direct_outputs[len(results)]\n                second = (\n                    symbolic_first\n                    if symbolic_first != first\n                    else symbolic_second\n                )\n                program_names = ("calibrated_model_prediction",) + names[:1]\n            else:\n                first = symbolic_first\n                second = symbolic_second\n                program_names = names\n            results.append(\n                SolveResult(\n                    first,\n                    second,\n                    program_names,\n                    len(programs),\n                    model_proposal_count,\n                    model_accepted_count,\n                    len(model_rejection_reasons),\n                    tuple(model_rejection_reasons),\n                    model_calibration_passed,\n                    model_direct_output_error,\n                )\n            )\n        elif model_direct_outputs is not None:\n            first = model_direct_outputs[len(results)]\n            second = test_input if test_input != first else first\n            results.append(\n                SolveResult(\n                    first,\n                    second,\n                    ("calibrated_model_prediction",),\n                    0,\n                    model_proposal_count,\n                    model_accepted_count,\n                    len(model_rejection_reasons),\n                    tuple(model_rejection_reasons),\n                    model_calibration_passed,\n                    model_direct_output_error,\n                )\n            )\n        else:\n            results.append(\n                SolveResult(\n                    test_input,\n                    test_input,\n                    ("fallback_identity",),\n                    0,\n                    model_proposal_count,\n                    model_accepted_count,\n                    len(model_rejection_reasons),\n                    tuple(model_rejection_reasons),\n                    model_calibration_passed,\n                    model_direct_output_error,\n                )\n            )\n    return results\n', 'submission.py': '"""Kaggle challenge-to-submission adapter and exact-match scoring helpers."""\n\nfrom __future__ import annotations\n\nfrom typing import Any, Callable\n\nfrom .grids import Grid, as_grid, to_json_grid\nfrom .inference import solve_task\nfrom .proposer import ProgramProposer\n\n\ndef make_submission(\n    challenges: dict[str, Any],\n    max_depth: int = 2,\n    proposer: ProgramProposer | None = None,\n    proposal_count: int = 3,\n    diagnostics: list[dict[str, Any]] | None = None,\n    progress_callback: Callable[[int, int, str], None] | None = None,\n) -> dict[str, list[dict[str, Any]]]:\n    submission: dict[str, list[dict[str, Any]]] = {}\n    for task_id, task in challenges.items():\n        predictions = solve_task(\n            task,\n            max_depth=max_depth,\n            proposer=proposer,\n            proposal_count=proposal_count,\n        )\n        if diagnostics is not None:\n            first = predictions[0]\n            diagnostics.append(\n                {\n                    "task_id": task_id,\n                    "consistent_programs": first.consistent_program_count,\n                    "model_proposals": first.model_proposal_count,\n                    "model_accepted": first.model_accepted_count,\n                    "model_rejected": first.model_rejected_count,\n                    "rejection_reasons": first.model_rejection_reasons,\n                    "model_calibration_passed": first.model_calibration_passed,\n                    "direct_output_error": first.model_direct_output_error,\n                }\n            )\n        submission[task_id] = [\n            {\n                "attempt_1": to_json_grid(prediction.attempt_1),\n                "attempt_2": to_json_grid(prediction.attempt_2),\n            }\n            for prediction in predictions\n        ]\n        if progress_callback is not None:\n            progress_callback(len(submission), len(challenges), task_id)\n    return submission\n\n\ndef exact_match_score(\n    submission: dict[str, list[dict[str, Any]]],\n    solutions: dict[str, Any],\n) -> tuple[int, int]:\n    """Return number of solved output grids and total output grids."""\n    solved = 0\n    total = 0\n    for task_id, expected_values in solutions.items():\n        predictions = submission[task_id]\n        if isinstance(expected_values, dict):\n            expected_values = expected_values.get("test", expected_values)\n        if not isinstance(expected_values, list) or len(expected_values) != len(predictions):\n            raise ValueError(f"Solution/prediction count mismatch for task {task_id}.")\n        for expected, prediction in zip(expected_values, predictions):\n            target: Grid = as_grid(expected)\n            total += 1\n            if (\n                as_grid(prediction["attempt_1"]) == target\n                or as_grid(prediction["attempt_2"]) == target\n            ):\n                solved += 1\n    return solved, total\n'}
for filename, source in SOURCES.items():
    (PACKAGE_DIR / filename).write_text(source, encoding='utf-8')
sys.path.insert(0, '/kaggle/working')


In [ ]:
import json
import time
from pathlib import Path
from collections import Counter
from arc_solver.local_model import LocalQwenProgramProposer, find_local_qwen_model
from arc_solver.submission import make_submission

input_files = list(Path('/kaggle/input').rglob('arc-agi_test_challenges.json'))
if len(input_files) != 1:
    raise FileNotFoundError(
        f'Expected exactly one arc-agi_test_challenges.json; found {len(input_files)}.'
    )
with input_files[0].open(encoding='utf-8') as stream:
    challenges = json.load(stream)
started = time.perf_counter()
model_path = find_local_qwen_model(Path('/kaggle/input'))
proposer = LocalQwenProgramProposer.from_pretrained(model_path)
diagnostics = []
def report_progress(completed, total, task_id):
    if completed == 1 or completed % 10 == 0 or completed == total:
        elapsed = time.perf_counter() - started
        print(f'Processed {completed}/{total} tasks in {elapsed / 60:.1f} minutes.', flush=True)
submission = make_submission(challenges, proposer=proposer, proposal_count=3, diagnostics=diagnostics, progress_callback=report_progress)
submission_path = Path('/kaggle/working/submission.json')
with submission_path.open('w', encoding='utf-8') as stream:
    json.dump(submission, stream, separators=(',', ':'))
proposal_count = sum(item['model_proposals'] for item in diagnostics)
accepted_count = sum(item['model_accepted'] for item in diagnostics)
rejected_count = sum(item['model_rejected'] for item in diagnostics)
calibrated_count = sum(item['model_calibration_passed'] for item in diagnostics)
reasons = Counter(reason for item in diagnostics for reason in item['rejection_reasons'])
direct_errors = Counter(item['direct_output_error'] for item in diagnostics if item['direct_output_error'])
print(f'Wrote {len(submission)} task predictions to {submission_path}')
print(f'Qwen generations: {len(diagnostics)}; candidate programs accepted: {accepted_count}; rejected: {rejected_count}; candidates: {proposal_count}')
print(f'Direct model predictions passed leave-one-out calibration: {calibrated_count}/{len(diagnostics)} tasks')
print(f'Program rejection reasons: {reasons.most_common(5)}')
print(f'Direct output issues: {direct_errors.most_common(5)}')
print(f'Total elapsed time including model loading: {(time.perf_counter() - started) / 60:.1f} minutes')
